<pre>
#+TITLE: Project 2: Analyzing the 2008 Financial Crisis
#+AUTHOR: Edgar Huamantla
#+DATE: 2026-03-22
</pre>

In [ ]:
# Because jupyter notebook has a global name space; use this area for imports
import pandas as pd
import numpy as np
import plotly.express as px
import math
from typing import Union
import re
from datetime import datetime

## SQLAlchemy imports
from sqlalchemy import create_engine, MetaData, Table, Column, DateTime as sqlalchemy_DateTime, Index, insert, text as sqlalchemy_text, func as sqlalchemy_func, select as sqlalchemy_select, event
from sqlalchemy import types as generic_sql_types
from sqlalchemy.dialects.sqlite import insert as sqlite_upsert

# aux packages
from IPython.display import Image, display, HTML
import logging
import pathlib
import importlib.util
import statsmodels.api as sm
import statistics

# set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# folder using CWD as base
curr_working_dir = pathlib.Path.cwd()
print(f"Current working directory: {curr_working_dir}")

image_folder = curr_working_dir / 'src' / 'images'
if not image_folder.exists():
    image_folder.mkdir(parents=True)
    logger.info(f"Created image folder at: {image_folder}")


### NOTE ON CONNECTION ENGINE
SQLITE:
- 3 slashes is relative paths
- 4 slashes is for absolute paths <br>
reference: https://docs.sqlalchemy.org/en/21/core/engines.html#sqlite

In [ ]:
# create a SQLit3 engine
db_file_path = curr_working_dir / 'barra_data.db'
engine = create_engine(f'sqlite:///{db_file_path}')

# KNOWN ISSUES
A hash function was not implemented to ensure duplicate records aren't entered.<br>
How that would work is that if a table is loaded, the payload is hashed and entered in a table existing in the database.<br>
Upon a subsequent load, it should be checked if the payload to INSERTED matches the hash value in the table containing previous payloads.<br>
If it does, then INSERT is not executed. <br>
## Why this matters
The reason is because we are running in a jupyter notebook, if you move around the cells, it creates the potential for double upload.<br>
We implemented a naive checker where we only look to see if dataframe record count matches the destination table record count.<br>
if it does, then the insert is skipped. If it doesn't, then it gets uploaded.
### Workaround
drop the table in question or delete the database. Unless you are being selective in which cell to run.

### AUX FUNCS

In [ ]:
def search_kw(keyword, dtype_dict):
    keyword = keyword
    matched_kw = [col for col in dtype_dict.keys() if keyword in col.lower()]
    return matched_kw

In [ ]:
class Std_dev_aggregate:
    def __init__(self):
        self.values = []
    
    def step(self, value):
        if value is not None:
            try:
                value = float(value)
                self.values.append(value)
            except (ValueError, TypeError):
                logger.warning(f"Non-numeric value encountered in STDDEV aggregate: {value}")
                pass  # Ignore non-numeric values

    def finalize(self):
        n = len(self.values)
        if n < 2:
            logger.warning("Not enough data points to compute standard deviation.")
            return None  # Not enough data to compute standard deviation
        return statistics.stdev(self.values)

In [ ]:
class Covar_samp_aggregate:
    def __init__(self):
        self.x_values = []
        self.y_values = []

    def step(self, x, y):
        if x is not None and y is not None:
            try:
                self.x_values.append(float(x))
                self.y_values.append(float(y))
            except (ValueError, TypeError):
                logger.warning(f"Non-numeric values encountered: x={x}, y={y}")
                pass

    def finalize(self):
        n = len(self.x_values)
        if n < 2:
            logger.warning("Not enough data points to compute covariance.")
            return None
        return statistics.covariance(self.x_values, self.y_values)

In [ ]:
class Median_aggregate:
    def __init__(self):
        self.values = []
    
    def step(self, value):
        if value is not None:
            try:
                self.values.append(float(value))
            except (ValueError, TypeError):
                logger.warning(f"Non-numeric value in MEDIAN aggregate: {value}")

    def finalize(self):
        if not self.values:
            return None
        return statistics.median(self.values)

In [ ]:
# event listener to inject STDDEV function into SQLite connection
@event.listens_for(engine, "connect")
def receive_connect(dbapi_connection, connection_record):
    dbapi_connection.create_aggregate("STDDEV", 1, Std_dev_aggregate)
    dbapi_connection.create_aggregate("COVAR_SAMP", 2, Covar_samp_aggregate)
    dbapi_connection.create_aggregate("MEDIAN", 1, Median_aggregate)

# Obtain resource files

In [ ]:


#check if gdown is installed in current python environment; alert the user to activate their virtual environment to avoid installing to global
if importlib.util.find_spec("gdown") is None:
    print("gdown is not installed. Please activate your virtual environment and install gdown to proceed.")

# dictionary: filename and gdown id
file_names = {'USE3L0712.RSK': '1Y0WE4cqbZfdNEvFuWIDR_d6oDbHHNkly',
              'USE3L0812.RSK': '1Z3vdd5m8uoB4whomq92DQbZHJiKYJc0v'}

for file_name, gdown_id in file_names.items():
    file_path = curr_working_dir / file_name
    if not file_path.exists():
        print(f"File {file_path} does not exist.")
        !gdown {gdown_id} -O {file_path}
    else:
        print(f"File {file_path} already exists. No need to download.")




In [ ]:
# check file integrity of downloaded files
for file_count, file_name in enumerate(file_names.keys(), start=1):
    print(f"\nChecking integrity of file {file_count}: {file_name}...")
    ! head -n 3 {file_name}

In [ ]:
# extract, parse text files into pandas df. ignore first row with date (i.e. 31DEC2007)
barra_df_07 = pd.read_csv('USE3L0712.RSK', skiprows=1, header=0)
barra_df_08 = pd.read_csv('USE3L0812.RSK', skiprows=1, header=0)

In [ ]:
# check shape of each dataframe
print(f"{barra_df_07.shape=}")
print(f"{barra_df_08.shape=}")
# assert columns match
assert barra_df_07.columns.equals(barra_df_08.columns), "Columns of the two dataframes do not match."
pd.testing.assert_index_equal(barra_df_07.columns, barra_df_08.columns, check_names=True) # secondary way using pd testing module

In [ ]:
display(barra_df_07.head(3))
display(barra_df_08.head(3))

In [ ]:
# look for null counts
for df in [barra_df_07, barra_df_08]:
    print(f"{df.isnull().values.any()=}")

## Cleaning Dataframe:
- remove white spaces from:
    - column names
    - string values in columns
    - index values

In [ ]:
def clean_index_whitespace(df):
    match df.index:
        case pd.Index() as idx if idx.dtype == 'O': # object dtype, likely string index
            df.index = df.index.str.strip()
            logger.info(f"stripped whitespace from {df.index.name=}")
        case _:
            logging.info(f"Index {idx.dtype=}. Skipping")
    return df

def clean_col_whitespace(df):
    match df.columns.dtype:
        case dtype if dtype == 'O':
            # create the stripped version
            original_columns = df.columns
            stripped_columns = original_columns.str.strip()

            # find the difference to output
            changed_cols_mask = original_columns != stripped_columns
            cleaned_cols = original_columns[changed_cols_mask].tolist()

            if cleaned_cols:
                logging.info(f"Stripped whitespace from columns: {cleaned_cols}")
            else:
                logging.info("No columns had leading/trailing whitespace to clean.")

            df.columns = stripped_columns
        case _:
            logging.info(f"skipped: {df.columns.dtype=} is not object dtype")
    return df


In [ ]:
# clean df index for both dataframes
barra_df_07 = clean_index_whitespace(barra_df_07)
barra_df_08 = clean_index_whitespace(barra_df_08)

# clean df columns for both dataframes
barra_df_07 = clean_col_whitespace(barra_df_07)
barra_df_08 = clean_col_whitespace(barra_df_08)

In [ ]:
# given a dataframe, and dict of invalid chars, replace invalid chars
def replace_invalid_chars_in_cols(df, invalid_char_dict): 
    for invalid_char, replacement in invalid_char_dict.items():
        df.columns = df.columns.str.replace(invalid_char, replacement, regex=False)
    return df

In [ ]:
def make_col_lower_case(df):
    df.columns = df.columns.str.lower()
    return df

In [ ]:
# apply replace_invalid_chars and make_col_lower_case to both dataframes
invalid_characters = {"%": "_pct"}
barra_df_07 = replace_invalid_chars_in_cols(barra_df_07, invalid_characters)
barra_df_08 = replace_invalid_chars_in_cols(barra_df_08, invalid_characters)

barra_df_07 = make_col_lower_case(barra_df_07)
barra_df_08 = make_col_lower_case(barra_df_08)

In [ ]:
# check df heads after cleaning
display(barra_df_07.columns)
display(barra_df_08.columns)

In [ ]:
# review data to see if it has been scaled, considering whether its a percentage
def review_numeric_cols_for_data_scaling(df, thresholds, exclude_pattern=None, precision=4):
    """
    looks at numeric columns in a dataframe.
    checks it against a provided threshold, client to provide. 
    Example: 
    thresholds = {'percentage': 150, 'std_dev': 5}
        if the column is a percentage, then a reasonable range would be -150 to 150.
        otherwise, we should expect to see between -5 and +5 standard deviations. 
    """
    findings = list()
    pd.options.display.float_format = f'{{:.{precision}f}}'.format # set float display precision for output

    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        col_min = round(df[col].min(), precision)
        col_max = round(df[col].max(), precision)

        is_pct_match = bool(exclude_pattern and re.search(exclude_pattern, col, re.IGNORECASE))

        match is_pct_match:
            case True:
                limit = thresholds.get('percentage', 150) # default to 150 if not specified
            case False:
                limit = thresholds.get('std_dev', 5) # default to 5 if not specified

        if col_max > limit or col_min < -limit:
            findings.append({
                'column': col,
                'min': col_min,
                'max': col_max,
                'limit_used': limit,
                'is_pct': is_pct_match,
                'precision': precision
            })
    return pd.DataFrame(findings)

In [ ]:
SCALING_THRESHOLDS = {'percentage': 200, 'std_dev': 10}

In [ ]:
# review numeric columns for data scaling, exclude percentage to review separately
num_findings_07 = review_numeric_cols_for_data_scaling(barra_df_07, SCALING_THRESHOLDS, exclude_pattern='_pct', precision=4)

print(f"Num columns with potential scaling issues (07):\n {num_findings_07[num_findings_07['is_pct'] == False]}")
print(f"Pct columns with potential scaling issues (07):\n {num_findings_07[num_findings_07['is_pct'] == True]}")



In [ ]:
num_findings_08 = review_numeric_cols_for_data_scaling(barra_df_08, SCALING_THRESHOLDS, exclude_pattern='_pct', precision=4)
print(f"Num columns with potential scaling issues (08):\n {num_findings_08[num_findings_08['is_pct'] == False]}")
print(f"Pct columns with potential scaling issues (08):\n {num_findings_08[num_findings_08['is_pct'] == True]}")

**Review for stocks with price = 0**

In [ ]:
print(f'# of rows with price less than or equal to 0: {barra_df_07[barra_df_07["price"] <= 0].shape[0]}')
print(f'# of rows with price less than or equal to 0: {barra_df_08[barra_df_08["price"] <= 0].shape[0]}')

Findings: <br>
- What we noticed was that ind1-5 do not appear to be meaningful numeric values. These are the sector mappings. 
- Price and Capitalization are in line with expectations. We were looking to see if there was negative values or setinel values to represent NULL.
- hbta does have a setinel value. -999 needs to be replaced with NULL.
- yld_pct in 07 is interesting as it has %50,450 max value that will be further investigated.

In [ ]:
def faceted_box_plots(
        df,
        y_column,
        facet_col,
        facet_col_wrap=4,
        base_row_height=300,
        template = 'plotly_dark',
        points = 'outliers',
        facet_col_spacing = 0.02,
        facet_row_spacing = 0.02,
        title = None
):
    # calc number of rows neeeded for facets 
    num_facets = df[facet_col].nunique()

    # avoid division by zero if only one facet
    facet_col_wrap = max(1, facet_col_wrap)
    num_rows = math.ceil(num_facets / facet_col_wrap)

    # adjust height
    total_height = base_row_height * num_rows

    fig = px.box(
        df,
        y=y_column,
        facet_col=facet_col,
        facet_col_wrap=facet_col_wrap,
        height=total_height,
        points=points,
        template=template,
        title=title,
        facet_col_spacing=facet_col_spacing,
        facet_row_spacing=facet_row_spacing
    )

    fig.update_yaxes(matches=None, showticklabels=True) # ensure each facet has its own y-axis range and shows tick labels

    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1])) # simplify facet labels to just the category value

    return fig


In [ ]:
col_to_include_07 = ['hbta', 'yld_pct']
df_07_filtered = barra_df_07[col_to_include_07]
df_07_filtered_melted = barra_df_07.melt(
    value_vars=col_to_include_07,
    var_name='factor_type',
    value_name='factor_value'
)
fig_07 = faceted_box_plots(
    df_07_filtered_melted,
    y_column='factor_value',
    facet_col='factor_type',
    title='Distribution of Numeric Factors with Potential Scaling Issues (07 Data)'
)
fig_07.show()

In [ ]:
# find the stock with yld_pct > 50K to investigate
outlier_yld_pct_07 = barra_df_07[barra_df_07['yld_pct'] > 50000]
outlier_yld_pct_07[['ticker', 'yld_pct', 'e3estu', 'price', 'capitalization']]

In our analysis, NEWCQ will be filtered out because it is not part of Barra's Estimation Universe. This is a penny stock so the outlier percentage makes sense.

In [ ]:
# 08 data
cols_to_include = ['hbta']
df_08_filtered = barra_df_08[cols_to_include]
df_08_filtered_melted = barra_df_08.melt(
    value_vars=cols_to_include,
    var_name='factor_type',
    value_name='factor_value'
)
fig_08 = faceted_box_plots(
    df_08_filtered_melted,
    y_column='factor_value',
    facet_col='factor_type',
    title='Distribution of Numeric Factors with Potential Scaling Issues (08 Data)'
)
fig_08.show()

In [ ]:
# get null counts for hbta column in both dataframes
hbta_null_count_07 = (barra_df_07['hbta'] == -999).sum()
hbta_null_count_08 = (barra_df_08['hbta'] == -999).sum()

print(f"Null count for hbta in 07 data: {hbta_null_count_07}")
print(f"Null count for hbta in 08 data: {hbta_null_count_08}")

In [ ]:
# replace the -999 with np.nan
barra_df_07['hbta'] = barra_df_07['hbta'].replace(-999, np.nan)
barra_df_08['hbta'] = barra_df_08['hbta'].replace(-999, np.nan)

In [ ]:
# plot 07 hbta distribution after replacing -999 with np.nan
fig = px.box(
    barra_df_07['hbta'],
    y='hbta',
    points='outliers',
    notched=True,
    template='plotly_dark',
    title='Box plot of hbta after replacing -999 with NaN (07 Data)'
)
fig.show()

In [ ]:
# plot 08 hbta distribution after replacing -999 with np.nan
fig = px.box(
    barra_df_08['hbta'],
    y='hbta',
    points='outliers',
    notched=True,
    template='plotly_dark',
    title='Box plot of hbta after replacing -999 with NaN (08 Data)'
)
fig.show()

# Database Schema Design
- Create at least 3 tables:
  - 2007 Barra USE3 data
  - 2008 Barra USE3 data  
  - Industry-to-Sector mapping (from PDF pages 83-84)
- Use the Industry-to-Sector table for all sector-related queries
- Address column naming issues (columns with '%')
- Optimize data types with justification
- Implement primary keys and foreign key relationships
- Create both single and composite indices

In [ ]:
# GIVEN
SECTORS = {
    'FINANCIAL': ['BANKS', 'THRIFTS', 'SECASSET', 'FINSVCS', 'PRPTYINS', 'LIFEINS'],
    'TECH_COMM': ['SEMICOND', 'CMPTRSW', 'CMPTRHW', 'INTERNET', 'INFOSVCS',
                  'ELECEQP', 'TELEPHON', 'WIRELESS', 'MEDIA', 'PUBLISH'],
    'REALESTATE_UTIL': ['EQTYREIT', 'ELECUTIL', 'GASUTIL'],
    'HEALTHCARE': ['MEDPRODS', 'BIOTECH', 'DRUGS', 'MEDPROVR'],
    'CONS_DISCR': ['SPLTYRET', 'RESTRNTS', 'CLOTHING', 'APPAREL', 'DEPTSTOR',
                   'HOTELS', 'ENTRTAIN', 'LEISURE', 'MOTORVEH', 'CONSDUR'],
    'CONS_STAPLES': ['GROCERY', 'FOODBEV', 'HOMEPROD', 'ALCOHOL', 'TOBACCO'],
    'INDUSTRIAL': ['CONSTRUC', 'INDPART', 'INDSVCS', 'HEAVYELC', 'HEAVYMCH',
                  'DEFAERO', 'ENVSVCS'],
    'BASIC_MAT': ['CHEMICAL', 'FOREST', 'GOLD'],
    'ENERGY': ['ENGYRES', 'OILSVCS', 'OILREF', 'MINING'],
    'TRANSPORT': ['AIRLINES', 'RAILROAD', 'TRUCKFRT']
}

In [ ]:
Image(image_folder / 'SECTOR_MAPPING_EXCERPT.jpg')

In [ ]:
# clean ticker column in both dataframes to remove whitespace and ensure consistent formatting
barra_df_07['ticker'] = barra_df_07['ticker'].str.strip()
barra_df_08['ticker'] = barra_df_08['ticker'].str.strip()

In [ ]:
# clean indname1 values for leading and trailing whitespace
barra_df_07['indname1'] = barra_df_07['indname1'].str.strip()
barra_df_08['indname1'] = barra_df_08['indname1'].str.strip()

In [ ]:
# get unique values for indname1 in both 07 and 08 to confirm sectors
unique_indname1_07 = barra_df_07['indname1'].unique()
unique_indname1_08 = barra_df_08['indname1'].unique()

In [ ]:
unique_indname1_07.sort()
unique_indname1_07

In [ ]:
# compare unique indname1 values to SECTORS dict values to confirm all sectors are represented
compare_df_to_sector_dict_07 = set(unique_indname1_07).difference({sector for sectors in SECTORS.values() for sector in sectors})
compare_df_to_sector_dict_08 = set(unique_indname1_08).difference({sector for sectors in SECTORS.values() for sector in sectors})

compare_sector_to_df_07 = set({sector for sectors in SECTORS.values() for sector in sectors}).difference(unique_indname1_07)
compare_sector_to_df_08 = set({sector for sectors in SECTORS.values() for sector in sectors}).difference(unique_indname1_08)

In [ ]:
display(f"Non matching: df -> sector dict 07: {compare_df_to_sector_dict_07}")
display(f"non matching: sector dict -> df 07: {compare_sector_to_df_07}")

In [ ]:
display(f"Non matching: df -> sector dict 08: {compare_df_to_sector_dict_08}")
display(f"non matching: sector dict -> df 08: {compare_sector_to_df_08}")

### ADDING SUFFIX to column names

In [ ]:
def add_suffix_to_cols(df, suffix):
    df.columns = [f"{col}_{suffix}" for col in df.columns]
    return df

In [ ]:
barra_df_07 = add_suffix_to_cols(barra_df_07, '07')
barra_df_08 = add_suffix_to_cols(barra_df_08, '08')

In [ ]:
# String Definitions, adding 50% buffer and rounding up with math.ceil
SQL_TYPE_MAPPING = {
    'int64': generic_sql_types.Integer,
    'float64': generic_sql_types.Float,
}

def generate_sqlalchemy_schema(meta_data, buffer = 1.5):
    """
    given a dictionary with col as keys and a nested dict with dtype and max length,
    we build a new dictionary that adds a 50% buffer to string lengths,
    if it is int64 or float64 we use a SQL_TYPE_MAPPING constants to use sqlalchemy types,
    we previously labeled columsn with values between 0 and 1 as boolean, so this handles it.  
    Buffer is added for future growth. 
    
    """
    
    orm_schema = dict()

    for col, info in meta_data.items():
        dtype = info['dtype']
        
        match dtype:
            case 'object':
                max_length = info['max_length']
                buffer_length = math.ceil(max_length * buffer) # add buffer% buffer and round up
                orm_schema[col] = generic_sql_types.String(buffer_length)
            case 'int64' | 'float64':
                orm_schema[col] = SQL_TYPE_MAPPING[dtype]
            case 'boolean':
                orm_schema[col] = generic_sql_types.Boolean
            case _:
                print(f"Data type {dtype} for column {col} not recognized.")
                
    return orm_schema


In [ ]:
def get_schema(df: pd.DataFrame) -> dict:
    """
    loop through a dataframe looking at each column and get its dtype name and length.
    """
    schema_dictonary = dict()
    for col in df.columns:
        dtype_name = df[col].dtype.name

        # create a nest dictionary for each column with dtype and length
        column_info = {"dtype": dtype_name}
        
        if dtype_name == 'object':
            # get the max length of strings in the column
            max_len = df[col].astype(str).str.len().max()
            column_info["max_length"] = int(max_len) if not pd.isna(max_len) else 0

        elif dtype_name in ['int64', 'float64']:
            unique_values = set(df[col].dropna().unique())
            if unique_values.issubset({0, 1}):
                column_info["dtype"] = 'boolean'

        schema_dictonary[col] = column_info
    return schema_dictonary
            

In [ ]:
type_dict_07 = get_schema(barra_df_07)
type_dict_08 = get_schema(barra_df_08)

In [ ]:
def get_string_max_lengths(type_dictionary):
    length_dict = dict()
    for col, info in type_dictionary.items():
        if info['dtype'] == 'object':
            length_dict[col] = info.get('max_length', 'N/A')
    return length_dict

In [ ]:
max_string_lengths_07 = get_string_max_lengths(type_dict_07)
max_string_lengths_08 = get_string_max_lengths(type_dict_08)
print(f"Max string lengths for 07 data:\n {max_string_lengths_07}")
print(f"Max string lengths for 08 data:\n {max_string_lengths_08}")


In [ ]:
# creating two dictionaries. note year has been added to suffix of columns earlier.
DTYPE_DICT_07 = generate_sqlalchemy_schema(type_dict_07) 
DTYPE_DICT_08 = generate_sqlalchemy_schema(type_dict_08) 

### Column Naming:
We decided to keep the column names the same, except where there were invalid characters.
The reason for this was that perhaps this columns are part of a standard data dictionary.
What we did decide to do was append the year to the columns so that we aren't reusing column names.
However, all new columns created by us would follow these rules:
- be given a name that suggests the purpose of the column and identify the characteristic of the stock to be represented
- Semantics:
    - the name should be meaningful and descriptive
    - clear and unambigous 
    - avoid acronyms
    - use abbreviations only if it enhances the column name
    - don't use a name that implicitly/explicitly identifies more than one characteristic.
    - use the singular form
    - not dynamic

### Column constraints:
We set the ticket to be the primary key for each table created.<br>
We set a constraint on CUSIP to be unique and not null.<br>

### Schema
The relationship we did establish was between the stocks and their risk factors to the sector mapping table.<br>
We chose the ticker to be the foreign key on the sector_mapping. <br>
We decided to add two columns, "created_on" and "updated_on".

### Additional Indexes
During the course of the project, we will review which columns are being most used to guide our decision in optimizing performance.<br>
For example, if we find ourselves ranking stocks by a risk factor, we will create an index on that column to speed up sorting.<br>
Another example would be if we find ourselves constantly looking for specific columns for a given stock, we will create a covered query -- a composite index, so that the database doesn't have to do a full scan.

In [ ]:
DTYPE_DICT_07

In [ ]:
def create_table_def_from_dtype_dict(dtype_dict):
    table_df = list()
    for col, dtype in dtype_dict.items():
        match col:
            case 'ticker':
                table_df.append(Column(col, dtype, primary_key=True))
            case 'cusip':
                table_df.append(Column(col, dtype, unique=True, nullable=False))
            case _:
                table_df.append(Column(col, dtype))
    return table_df

In [ ]:
table_07 = create_table_def_from_dtype_dict(DTYPE_DICT_07)
table_08 = create_table_def_from_dtype_dict(DTYPE_DICT_08)
print(f"{table_07[:2]=}")
print(f"{table_08[:2]=}")

**NOTE**: CUSIP doesn't show UNIQUE constraint because it's created in the background. We will check this after association with a metadata object.

**NOTE**: Because industry will be commonly searched on, we will create a covered query (a composite index) between Industry and sector. <br>
I.e: 'BANKS' --> 'FINANCIAL'

In [ ]:
# find max length of a string in a dictionary
max_length_string_in_sector_dict= 0 
for sector, industries in SECTORS.items():
    if len(sector) > max_length_string_in_sector_dict:
        max_length_string_in_sector_dict = len(sector)
    for industry in industries:
        if len(industry) > max_length_string_in_sector_dict:
            max_length_string_in_sector_dict = len(industry)
print(f"Max string length across all sectors and industries: {max_length_string_in_sector_dict}")

In [ ]:
flattened_industry_to_sector_mapping = list()
for sector, industries in SECTORS.items():
    for industry in industries:
        flattened_industry_to_sector_mapping.append(
            (industry, sector)
        )
flattened_industry_to_sector_mapping_df = pd.DataFrame(flattened_industry_to_sector_mapping, columns=['industry', 'sector'])
print(f"Shape of flattened industry to sector mapping df: {flattened_industry_to_sector_mapping_df.shape}")
flattened_industry_to_sector_mapping_df.head(5)

In [ ]:
# review dfs to be inserted
display(barra_df_07.head(1))
display(barra_df_08.head(1))
display(flattened_industry_to_sector_mapping_df.head(1))

### Potential Problem
Database file may already have been created and populated with data. <br>
Two approaches:
1. check if record counts in passed df match record count in destination table. It provides some protection.
2. create a table and store table_name, row_count, and a df_hash. before inserting, hash df and compare.

In [ ]:
def bulk_upsert_dataframe_into_sqlite(engine, table_object, df, chunk_size=1000):
    """
    Focused Upsert: Assumes schema exists. Uses ON CONFLICT DO NOTHING 
    to ensure idempotency at the row level.
    """
    table_name = table_object.name
    records = df.to_dict(orient='records')
    
    logger.info(f"Loading {len(records)} records into {table_name}...")
    
    for i in range(0, len(records), chunk_size):
        chunk = records[i:i + chunk_size]
        
        # SQLite-specific upsert: ignore if PK (ticker) already exists
        stmt = sqlite_upsert(table_object).values(chunk).on_conflict_do_nothing()
        
        with engine.begin() as conn:
            conn.execute(stmt)
            
    logger.info(f"Upsert into {table_name} complete.")

In [ ]:
metadata = MetaData()

table_barra_07 = Table(
    'barra_07',
    metadata,
    *table_07,
    Column('created_on', sqlalchemy_DateTime(), default=datetime.now),
    Column('updated_on', sqlalchemy_DateTime(), default=datetime.now, onupdate=datetime.now),

    # later added barra_id index
    Index('idx_barraid_07', 'barrid_07')
)

table_barra_08 = Table(
    'barra_08',
    metadata,
    *table_08,
    Column('created_on', sqlalchemy_DateTime(), default=datetime.now),
    Column('updated_on', sqlalchemy_DateTime(), default=datetime.now, onupdate=datetime.now),

    # later added barra_id index
    Index('idx_barraid_08', 'barrid_08')

)

buffer_length_for_industry_sector = math.ceil(max_length_string_in_sector_dict * 1.5)
table_industry_sector_mapping = Table(
    'industry_to_sector',
    metadata,
    Column('industry', generic_sql_types.String(buffer_length_for_industry_sector), primary_key=True),
    Column('sector', generic_sql_types.String(buffer_length_for_industry_sector), nullable=False),
    Column('created_on', sqlalchemy_DateTime(), default=datetime.now),
    Column('updated_on', sqlalchemy_DateTime(), default=datetime.now, onupdate=datetime.now),

    # create a covered index for performance
    Index('idx_industry_sector', 'industry', 'sector')
)

metadata.create_all(engine)

### ADDING HACK TO PREVENT DUPE INSERTS
Because of the earlier disclosed issue and not making a hash of the data inserted or a full DB scan.
we are going to simply check if the db file exists and the tables have already abeen created.
#### New issues introduced
What will happen is that some queries will fail or return blanks because db has been modified afterwards.
Clear/Restart Kernel and then re-run. But queries that worked the first time before DB was modified will break.

In [ ]:
db_exists = db_file_path.exists()
load_required = False

if db_exists:
    # 2. Schema Check: Are the tables actually there?
    from sqlalchemy import inspect
    inspector = inspect(engine)
    existing_tables = inspector.get_table_names()

    required_tables = ['barra_07', 'barra_08', 'industry_to_sector']
    if all(table in existing_tables for table in required_tables):
        with engine.connect() as connection:
            count = connection.execute(sqlalchemy_text("SELECT COUNT(*) FROM barra_07")).scalar()
            logger.info(f"Row count in barra_07: {count}")
        if count > 0:
            logger.info("Data found in barra_07. Skipping initial load.")
        else:
            logger.info("Tables exist but no data found. Proceeding to load.")
            load_required = True
    
else:
    logger.info("Database file not found. Creating and loading...")
    load_required = True

if load_required:
    bulk_upsert_dataframe_into_sqlite(engine, table_barra_07, barra_df_07)
    bulk_upsert_dataframe_into_sqlite(engine, table_barra_08, barra_df_08)
    bulk_upsert_dataframe_into_sqlite(engine, table_industry_sector_mapping, flattened_industry_to_sector_mapping_df)

In [ ]:
# CHECKING SQL DB if record count matches df with SQL query
def check_record_count_in_sqlite(engine, table_object, df):
    # get names for logging
    db_path = pathlib.Path(engine.url.database) if engine.url.database else None
    db_name = db_path.name if db_path else 'in-memory database'
    table_name = table_object.name

    with engine.connect() as connection:
        count_before = connection.execute(
            sqlalchemy_select(sqlalchemy_func.count()).select_from(table_object)
            ).scalar() or 0

    if count_before == len(df):
        logger.info(f"table {table_name} count matches df.")
        return


In [ ]:
# check record count in sqlite matches df record count for each table
list_of_tables_created = ['barra_07', 'barra_08', 'industry_to_sector']
df_list = [barra_df_07, barra_df_08, flattened_industry_to_sector_mapping_df]
_ = [
    check_record_count_in_sqlite(engine, table,df)
    for table, df in zip([table_barra_07, table_barra_08, table_industry_sector_mapping], df_list)
]

### Stocks that disappeared in 2008
From our results:
- BUD: Anheuser-Busch Acquired and merged with InBev. Nov 2008 – Sept 2009: The symbol was briefly inactive on the NYSE while the new combined company, Anheuser-Busch InBev, traded as ABI in Brussels.
- NWS.A: News Corp (NWS) Merged with Dow Jones and Company. 
- LEH: Lehman Brothers 

How we are going to handle:
- FIRST because these cells might be ran again (we are in a juyptyer notebook), we will first check if our changes below have been commited.
- We want drop BUD from the 2007 table. if that succeeeded then update nwsa in 2008 to nws.a<br>
Because we are doing writes to the database, we are okay with the overhead added by engine.connect().begin()

In [ ]:
# for reference, find stocks that disappeared from our universe
query_stocks_in_08_not_in_07_estu = sqlalchemy_text(
"""
SELECT
    a.name_07,
    a.ticker_07,
    a.nonestu_07 as nonestu_07,
    b.nonestu_08 as nonestu_08,
    a.capitalization_07/1e9 AS cap_billions_07,
    a.size_07,
    a.srisk_pct_07,
    b.srisk_pct_08,
    a.trisk_pct_07,
    b.trisk_pct_08
FROM
    barra_07 AS a
LEFT JOIN
    barra_08 AS b
ON a.ticker_07 = b.ticker_08
WHERE a.nonestu_07 = 0
AND a.size_07 > 0
AND (a.ticker_07 IS NOT NULL and b.price_08 IS NULL);
"""    
)
with engine.connect() as connection:
    results_stocks_in_08_not_in_07_estu = pd.DataFrame(connection.execute(query_stocks_in_08_not_in_07_estu).fetchall())
results_stocks_in_08_not_in_07_estu

In [ ]:
# look for bear stearns
query_bear_stearns = sqlalchemy_text(
"""
SELECT
    a.name_07 as name_07,
    a.ticker_07 as ticker_07,
    b.ticker_08 as ticker_08,
    b.name_08 as name_08,
    a.capitalization_07/1e9 as capitalization_07_billions,
    b.capitalization_08/1e9 as capitalization_08_billions,
    a.size_07 as size_07,
    a.nonestu_07 as nonestu_07,
    a.srisk_pct_07 as srisk_07,
    b.srisk_pct_08 as srisk_08,
    a.trisk_pct_07 as trisk_07,
    b.trisk_pct_08 as trisk_08
FROM
    barra_07 AS a
LEFT JOIN
    barra_08 AS b
ON a.ticker_07 = b.ticker_08
WHERE a.name_07 = 'BSC' OR a.name_07 LIKE 'BEAR%' or a.ticker_07 LIKE "NWS%";
"""
)
with engine.connect() as connection:
    results_query_bear_stearns = pd.DataFrame(connection.execute(query_bear_stearns).fetchall())
results_query_bear_stearns

In [ ]:
# check to see if bush and news corp are in 2008 data
query_checking_for_bud_and_news_corp_in_08 = sqlalchemy_text(
"""
SELECT
    ticker_08,
    name_08,
    capitalization_08/1e9 as capitalization_08_billions,
    nonestu_08,
    size_08
FROM
    barra_08
WHERE name_08 LIKE 'ANHEU%' OR name_08 LIKE 'NEWS%'
"""
)
with engine.connect() as connection:
    results_checking_for_bud_and_news_corp_in_08 = pd.DataFrame(connection.execute(query_checking_for_bud_and_news_corp_in_08).fetchall())
results_checking_for_bud_and_news_corp_in_08

In [ ]:
query_check_if_drop_and_update_occured_for_bud_and_nwsa = sqlalchemy_text(
"""
SELECT
    EXISTS(
        SELECT 1
        FROM barra_07
        WHERE ticker_07 = 'BUD') AS needs_bud_drop,
    EXISTS(
        SELECT 1
        FROM barra_08
        WHERE ticker_08 = 'NWSA') AS needs_nwsa_update
"""
)
with engine.connect() as connection:
    result = connection.execute(query_check_if_drop_and_update_occured_for_bud_and_nwsa).fetchone()
    logger.info(f"created row object: result[{result._fields}]")
    logger.info(f"row values: {result}")
logger.info(f"Needs NWSA update: { 'TRUE' if result.needs_nwsa_update else 'FALSE' }")
logger.info(f"Needs BUD drop: { 'TRUE' if result.needs_bud_drop else 'FALSE' }")

current_state_check = result._asdict()
logger.info(f"converted row to dict: {current_state_check}")

match current_state_check:
    case {"needs_bud_drop": 1, "needs_nwsa_update": 1}:
        logger.info("All actions need to be performed.")
        sql_statements = [
            "DELETE FROM barra_07 WHERE ticker_07 = 'BUD';",
            "SAVEPOINT bud_dropped;",
            "UPDATE barra_08 SET ticker_08 = 'NWS.A' WHERE ticker_08 = 'NWSA';",
            "SAVEPOINT news_corp_updated;",
        ]
        with engine.begin() as connection:
            for stmt in sql_statements:
                connection.execute(sqlalchemy_text(stmt))
                logger.info(f"executing {stmt=}")
            logger.info("Executed BUD drop and NWSA update in a single transaction with savepoints for safety.")

    case {"needs_bud_drop": 1, "needs_nwsa_update": 0}:
        logger.info("Preparing to DROP BUD from barra_07.")
        sql_payload = sqlalchemy_text("DELETE FROM barra_07 WHERE ticker_07 = 'BUD';")
        with engine.begin() as connection:
            connection.execute(sql_payload)
            logger.info("Executed BUD drop from barra_07.")
    
    case {"needs_bud_drop": 0, "needs_nwsa_update": 1}:
        logger.info("Preparing to UPDATE NWSA to NWS.A in barra_08.")
        sql_payload = sqlalchemy_text("UPDATE barra_08 SET ticker_08 = 'NWS.A' WHERE ticker_08 = 'NWSA';")
        with engine.begin() as connection:
            connection.execute(sql_payload)
            logger.info("Executed NWSA update in barra_08.")
    case {"needs_bud_drop": 0, "needs_nwsa_update": 0}:
        logger.info("No action needed. BUD already dropped and NWSA already updated.")

In [ ]:
# check if bud really exists in table_barra_07
query_check_bud_existence = sqlalchemy_text(
"""
SELECT
    ticker_07,
    name_07
FROM barra_07
WHERE ticker_07 = 'BUD';
"""
)
with engine.connect() as connection:
    results_check_bud_existence = pd.DataFrame(connection.execute(query_check_bud_existence).fetchall())
results_check_bud_existence

### 2. Risk Metrics Analysis (25%)
**Key Decisions Required:**

**Beta Comparison:**
- Compare 'HBTA' vs 'BETA' columns
- Research differences in the Barra handbook
- Recommend which the risk management division should use with justification



In [ ]:
Image(image_folder / 'BETA_HBETA_Def_Excerpt-barra_handbook.jpg')

In [ ]:
beta_search = search_kw('ta', DTYPE_DICT_07)
beta_search

In [ ]:
# build query, simple average of beta where nonestu =0 and size>0. 
# then get weighted average by capitalization where nonestu = 0 and size > 0.

simple_and_weighted_average_of_beta_and_hbeta_07 = sqlalchemy_text(
"""
SELECT
    AVG(beta_07) AS simple_avg_beta,
    SUM(beta_07 * capitalization_07) / SUM(capitalization_07) AS weighted_avg_beta,
    AVG(hbta_07) AS simple_avg_hbta,
    SUM(hbta_07 * capitalization_07) / SUM(capitalization_07) AS weighted_avg_hbta
FROM
    barra_07
WHERE
    nonestu_07 = 0 AND size_07 > 0;
"""
)
simple_and_weighted_average_of_beta_and_hbeta_08 = sqlalchemy_text(
"""
SELECT
    AVG(beta_08) AS simple_avg_beta,
    SUM(beta_08 * capitalization_08) / SUM(capitalization_08) AS weighted_avg_beta,
    AVG(hbta_08) AS simple_avg_hbta,
    SUM(hbta_08 * capitalization_08) / SUM(capitalization_08) AS weighted_avg_hbta
FROM
    barra_08
WHERE
    nonestu_08 = 0 AND size_08 > 0;
"""
)
with engine.connect() as connection:
    result_07 = pd.DataFrame([connection.execute(simple_and_weighted_average_of_beta_and_hbeta_07).fetchone()])
    result_08 = pd.DataFrame([connection.execute(simple_and_weighted_average_of_beta_and_hbeta_08).fetchone()])

print(f"07 Beta Averages:\n {result_07}")
print(f"08 Beta Averages:\n {result_08}")
dif_results = result_08 - result_07
print(f"Difference in Beta Averages (08 - 07):\n {dif_results}")


In [ ]:
query_stock_beta_market_return = sqlalchemy_text(
"""
SELECT
    a.ticker_07 AS ticker,
    a.beta_07 AS beta,
    a.hbta_07 AS historical_beta,
    m.sector AS sector,
    (a.capitalization_07 / 1e9) AS cap_billions_07,
    ((b.price_08 - a.price_07) / a.price_07) AS realized_return_08
FROM
    barra_07 AS a
JOIN
    barra_08 AS b ON a.ticker_07 = b.ticker_08
JOIN
    industry_to_sector AS m ON a.indname1_07 = m.industry
WHERE a.nonestu_07 = 0
AND a.size_07 > 0
AND b.nonestu_08 = 0
AND b.size_08 > 0;
"""
)
with engine.connect() as connection:
    stock_beta_market_return_df = pd.DataFrame(connection.execute(query_stock_beta_market_return).fetchall(), columns=['ticker', 'beta', 'historical_beta', 'sector', 'cap_billions_07', 'realized_return_08'])

In [ ]:
common_kwargs = {
    'y': 'realized_return_08',
    'size': 'cap_billions_07',
    'color': 'sector',
    'hover_name': 'ticker',
    'size_max': 50,
    'opacity': 0.7,
    'trendline': 'ols',
    'template': 'plotly_dark',
    'labels':{
        'realized_return_08': 'Realized Return (08 vs 07)',
        "cap_billions_07": "Market Cap (Billions, 07)",
    }
}

# beta vs return
fig_beta = px.scatter(
    stock_beta_market_return_df,
    x='beta',
    title='Predicted Beta (07) vs Realized Return (08 vs 07)',
    **common_kwargs
)
fig_beta.add_hline(y=0, line_dash='dash', line_color='red')
fig_beta.show()

In [ ]:
# historical beta plot vs return (08-07)
fig_hbta = px.scatter(
    stock_beta_market_return_df,
    x='historical_beta',
    title='Historical Beta (07) vs Realized Return (08 vs 07)',
    **common_kwargs
)
fig_hbta.add_hline(y=0, line_dash='dash', line_color='red')
fig_hbta.show()

In [ ]:
def get_risk_metric_stats(df, risk_col, target_col):
    X = sm.add_constant(df[risk_col])
    y = df[target_col]
    
    model = sm.OLS(y, X).fit()
    
    return {
        "R-Squared": model.rsquared,
        "RMSE": (model.mse_resid)**0.5,
        "P-Value": model.pvalues[risk_col]
    }

In [ ]:
# Run the comparison
stats_predicted = get_risk_metric_stats(stock_beta_market_return_df, 'beta', 'realized_return_08')
stats_historical = get_risk_metric_stats(stock_beta_market_return_df, 'historical_beta', 'realized_return_08')

# Print a clean comparison table
print("Risk Metric Performance (2008 Crisis):")
print(pd.DataFrame([stats_predicted, stats_historical], 
                   index=['Beta', 'HBeta']))

**Risk Measure Comparison:**
- Compare 'SRISK%' (SYSTEMATIC RISK) vs 'TRISK%'
- Research differences in the Barra handbook
- Recommend which measure for risk assessment with justification

**DEFINITIONS** From barra handbook pdf
1. Systematic Risk: The risk (annualized standard deviation) of the systematic return.
2. Systematic Return: The part of the return dependent on the benchmark return. <br>
We can break excess returns into two components: systematic and residual. The systematic return is the beta times the benchmark excess return.
3. Benchmark: A reference portfolio for active management. The goal of the active manager is to exceed the benchmark return.
4. Total Risk: breakdown into specific and common factor risk. There is no concept of a market, or systematic, portfolio. <br>
The risk is attributed purely to common factor and security-specific influences.
    - Common factor risk: is portfolio risk that arises from assets’ exposures to common factors, such as capitalization and industries.
    - Specific risk: is unique to a particular company and thus is uncorrelated with the specific risk of other assets. <br>
    For a portfolio, specific risk is the weighted sum of all the holdings’ specific risk values.
5. NONESTU: Indicator for firms outside US-E3 estimation universe. <br>
This is a 0-1 indicator variable: It is equal to 0 if the company is in the US-E3 estimation universe and equal to 1 if the company is outside the US-E3 estimation universe.

In [ ]:
search_kw('risk', DTYPE_DICT_07)

In [ ]:
query_for_scatter_correlation_trisk_srisk = sqlalchemy_text(
"""
SELECT
    a.ticker_07 AS ticker,
    ((b.price_08 - a.price_07) / a.price_07) AS realized_return_08,
    a.trisk_pct_07 AS trisk_pct_07,
    a.srisk_pct_07 AS srisk_pct_07
FROM
    barra_07 AS a
JOIN
    barra_08 AS b ON a.ticker_07 = b.ticker_08
WHERE a.nonestu_07 = 0
AND a.size_07 > 0
AND b.nonestu_08 = 0
AND b.size_08 > 0;
"""
)
with engine.connect() as connection:
    result_scatter_trisk_srisk = pd.DataFrame(connection.execute(query_for_scatter_correlation_trisk_srisk).fetchall())
result_scatter_trisk_srisk

In [ ]:
common_kwargs_scatter = {
    'y': 'realized_return_08',
    'hover_name': 'ticker',
    'opacity': 0.7,
    'trendline': 'ols',
    'template': 'plotly_dark',
    'labels':{
        'realized_return_08': 'Realized Return (08 vs 07)',
        'trisk_pct_07': 'Total Risk (%)',
        'srisk_pct_07': 'Systematic Risk (%)'
    }
}

fig_total_risk = px.scatter(
    result_scatter_trisk_srisk,
    x='trisk_pct_07',
    title='Total Risk (%) (07) vs Realized Return (08 vs 07)',
    **common_kwargs_scatter
)
fig_total_risk.add_hline(y=0, line_dash='dash', line_color='red')
fig_total_risk.show()

fig_systematic_risk = px.scatter(
    result_scatter_trisk_srisk,
    x='srisk_pct_07',
    title='Systematic Risk (%) (07) vs Realized Return (08 vs 07)',
    **common_kwargs_scatter
)
fig_systematic_risk.add_hline(y=0, line_dash='dash', line_color='red')
fig_systematic_risk.show()

# run comparison Total risk and Systematic Risk
stats_total_risk = get_risk_metric_stats(result_scatter_trisk_srisk, 'trisk_pct_07', 'realized_return_08')
stats_systematic_risk = get_risk_metric_stats(result_scatter_trisk_srisk, 'srisk_pct_07', 'realized_return_08')

print("Risk Metric Performance vs Realized Return:")
print(pd.DataFrame([stats_total_risk, stats_systematic_risk],
                   index=['Total Risk', 'Systematic Risk']))

**Risk Prediction Model:**
Using SQL, calculate mean and standard deviation for potential risk indicators:
- TRADEACT (trading activity)
- VOLTILTY (volatility)
- MOMENTUM
- VALUE
- EARNVAR (earnings variability)
- LEVERAGE


In [ ]:
kws = ['trade', 'vol', 'mom', 'val', 'earnvar', 'lev']
search_res = list(
    map(lambda kw: search_kw(kw, type_dict_07), kws))
search_res


In [ ]:
query_07_calculate_avg_std_for_tradeact_vol_mom_val_earnvar_lev = sqlalchemy_text(
"""
SELECT
    AVG(tradeact_07) AS avg_tradeact_07,
    AVG(voltilty_07) AS avg_volatility_07,
    AVG(momentum_07) AS avg_mom_07,
    AVG(value_07) AS avg_value_07,
    AVG(earnvar_07) AS avg_earnvar_07,
    AVG(leverage_07) AS avg_lev_07,
    STDDEV(tradeact_07) AS std_tradeact_07,
    STDDEV(voltilty_07) AS std_volatility_07,
    STDDEV(momentum_07) AS std_mom_07,
    STDDEV(value_07) AS std_value_07,
    STDDEV(earnvar_07) AS std_earnvar_07,
    STDDEV(leverage_07) AS std_lev_07
FROM
    barra_07
WHERE nonestu_07 = 0 AND size_07 > 0;
"""
)
with engine.connect() as connection:
    result_avg_std = pd.DataFrame([connection.execute(query_07_calculate_avg_std_for_tradeact_vol_mom_val_earnvar_lev).fetchone()])

result_avg_std


### Determine which factors (individually or in combination) best predict PCT_CHANGE between 2007-2008.

In [ ]:
search_kw('sap', type_dict_07)

Experiment setup:

In [ ]:
# lets see if given factors data is z-scored normed against SPY500; selecting volatility as an example factor
query_to_check_if_factors_are_z_scored_normed_sap500 = sqlalchemy_text(
"""
SELECT
    AVG(voltilty_07) AS avg_volatility_07,
    STDDEV(voltilty_07) AS std_volatility_07
FROM
    barra_07
WHERE sap500_07 = 1
"""
)
with engine.connect() as connection:
    results_query_to_check_if_factors_are_z_scored_normed_sap500 = pd.DataFrame([connection.execute(query_to_check_if_factors_are_z_scored_normed_sap500).fetchone()])

results_query_to_check_if_factors_are_z_scored_normed_sap500

In [ ]:
query_to_check_if_factors_are_z_scored_normed_sap500_nonestu = sqlalchemy_text(
"""
SELECT
    AVG(voltilty_07) AS avg_volatility_07,
    STDDEV(voltilty_07) AS std_volatility_07
FROM
    barra_07
WHERE sap500_07 = 1 AND nonestu_07 = 0;
"""
)
with engine.connect() as connection:
    results_query_to_check_if_factors_are_z_scored_normed_sap500_nonestu = pd.DataFrame([connection.execute(query_to_check_if_factors_are_z_scored_normed_sap500_nonestu).fetchone()]) 
results_query_to_check_if_factors_are_z_scored_normed_sap500_nonestu

In [ ]:
query_to_check_if_factors_are_z_scored_normed_nonestu = sqlalchemy_text(
"""
SELECT
    AVG(voltilty_07) AS avg_volatility_07,
    STDDEV(voltilty_07) AS std_volatility_07
FROM
    barra_07
WHERE nonestu_07 = 0;
"""
)
with engine.connect() as connection:
    results_query_to_check_if_factors_are_z_scored_normed_nonestu = pd.DataFrame([connection.execute(query_to_check_if_factors_are_z_scored_normed_nonestu).fetchone()]) 
results_query_to_check_if_factors_are_z_scored_normed_nonestu

In [ ]:
query_to_check_if_factors_are_z_scored_normed_nonestu_size = sqlalchemy_text(
"""
SELECT
    AVG(voltilty_07) AS avg_volatility_07,
    STDDEV(voltilty_07) AS std_volatility_07
FROM
    barra_07
WHERE nonestu_07 = 0 AND size_07 > 0;
"""
)
with engine.connect() as connection:
    results_query_to_check_if_factors_are_z_scored_normed_nonestu_size = pd.DataFrame([connection.execute(query_to_check_if_factors_are_z_scored_normed_nonestu_size).fetchone()])
results_query_to_check_if_factors_are_z_scored_normed_nonestu_size

We decided to review our factors to see how they were normalized to interpret our results<br>

```text.org

FACTOR = Volatility
|---+--------------------+----------+---------|
| # | QUERY              |  AVG_VOL | STD_VOL |
|---+--------------------+----------+---------|
| 1 | IN SAP500          |   0.2358 |  0.8738 |
| 2 | IN SAP500 + ESTU   |   0.2358 |  0.8738 |
| 3 | IN ESTU            |   0.6957 |  1.0000 |
| 4 | IN ESTU + SIZE > 0 | (0.1010) |  0.6082 |
|---+--------------------+----------+---------|
```

The factors do not appear to normalized against the SAP500. <br>
- Instead they appear to be normalized by the Estimation Universe (nonestu = 0) because the standard deviation is exactly 1.
- It appears the mean was normalized to a different base or used a weighted approach. However the mean approached 0 when we checked for SAP500 membership, suggesting a cap-weighted approach. 
- An interesting result was when we did membership in the estimation universe and size factor greater than 0, our mean was near 0 [-.10] and std_dev .60.

Our experiment will filter out companies with size less than 0. This won't change the objectives of our results to see if R-Squared is improved with the combinations of fundamental and technical factors. However, our unit of scale in our working universe is 1.67 (adjusted for 1/.60)

### Needs to be derived

1. Stock return
2. Market Return 
3. Residual Return
4. Residual Risk
5. Industry Dummy Variables

#### Special Considerations
LEFT JOINS -- BARRA_07 --> BARRA_08. So that stocks that existed in 2007 are included in results. <br>
Handled by giving NULL values (because they are missing) as (100)% return. <br>
    - **NOTE** DF was checekd earlier before loading to see if any stock had a price lower or equal to 0. 

In [ ]:
# stock return from 08-07 for stocks in nonestu = 0 and size > 0
# previously we were using multiple queries, 1) stock returns and setting delisted companies to -1.0 return 
# then 2) getting market cap weighted average return. 3) then residual returns.
# we are combining now with a window function to ensure same universe is used.
query_for_stock_market_systematic_residual_returns = sqlalchemy_text(
"""
WITH Returns AS(
    SELECT
        a.ticker_07 AS ticker,
        a.beta_07 AS beta,
        a.capitalization_07 AS capitalization_07,
        CASE
            WHEN b.price_08 IS NULL THEN -1.0
            ELSE ((b.price_08 - a.price_07) / a.price_07)
        END AS stock_return
    FROM
        barra_07 AS a
    LEFT JOIN
        barra_08 AS b ON a.ticker_07 = b.ticker_08
    WHERE a.nonestu_07 = 0
    AND a.size_07 > 0
),
Market_Calc AS (
SELECT
    *,
    --- explicit about weight for each stock
    (capitalization_07 * 1.0)/ (SUM(capitalization_07) OVER ()) AS stock_weight,
    (SUM(stock_return * capitalization_07) OVER ()) / (SUM(capitalization_07) OVER ()) AS market_cap_weighted_returns
FROM Returns
)
SELECT
    ticker,
    stock_return,
    stock_weight,
    market_cap_weighted_returns,
    beta,
    --- systematic returns calculation
    (beta * market_cap_weighted_returns) AS systematic_return,
    -- Residual Return
    (stock_return - (beta * market_cap_weighted_returns)) AS residual_return
FROM Market_Calc
"""
)
with engine.connect() as connection:
    result_query_for_stock_market_systematic_residual_returns = pd.DataFrame(connection.execute(query_for_stock_market_systematic_residual_returns).fetchall())
print(f'{result_query_for_stock_market_systematic_residual_returns.shape=}')
display(result_query_for_stock_market_systematic_residual_returns)
assert math.isclose(result_query_for_stock_market_systematic_residual_returns['stock_weight'].sum(), 1.0, rel_tol=1e-5), "Stock weights do not sum to 1"

In [ ]:
# risk decomposition verification
# the relationship must hold, where trisk = square_root(systematic_risk**2) + square_root(residual_risk**2) [ total_variance = systematic_variance + residual_variance ]
query_risk_decomposition_verification = sqlalchemy_text(
"""
WITH Risk_Decomp AS(
    SELECT
        a.ticker_07 AS ticker,
        a.name_07 AS name,
        a.indname1_07 AS industry,
        a.trisk_pct_07 AS total_risk_pct, --- given from the data
        a.srisk_pct_07 AS systematic_risk_pct, --- given from the data
        --- calculate residual risk: sqrt(total**2 - systematic_risk**2)
        SQRT(POWER(a.trisk_pct_07, 2) - POWER(a.srisk_pct_07, 2)) AS calculated_residual_risk_pct,
        a.beta_07 AS beta,
        a.capitalization_07 AS capitalization_07
    FROM barra_07 AS a
    JOIN industry_to_sector AS s ON a.indname1_07 = s.industry
    WHERE a.nonestu_07 = 0
    AND a.size_07 > 0
)
SELECT
    *,
    -- verification of risk decomposition: total_risk should be approximately equal to sqrt(systematic_risk^2 + residual_risk^2)
    SQRT(POWER(systematic_risk_pct, 2) + POWER(calculated_residual_risk_pct, 2)) AS reconstructed_total_risk,
    -- use boolean to check if the reconstructed total risk is close to the given total risk within a small tolerance
    CASE
        WHEN ABS(total_risk_pct - SQRT(POWER(systematic_risk_pct, 2) + POWER(calculated_residual_risk_pct, 2))) < 1e-5 THEN 1
        ELSE 0
    END AS is_risk_identity_valid
FROM Risk_Decomp;
"""
)
with engine.connect() as connection:
    result_risk_decomposition_verification = pd.DataFrame(connection.execute(query_risk_decomposition_verification).fetchall())
print(f'{result_risk_decomposition_verification.shape=}')
display(result_risk_decomposition_verification.head(5))

### CHANGE IN DESIGN
Because we are using a filter to obtain our universe (stock is in the estimation universe and it's size is typical) we might have industries with small number of stocks. Given that our shape of universe is about 96 stocks.<br>
As a result, we are going to use sectors. Another reasoning is the same systemic risks impacting a sector are likely to have impacted its industry composition. <br>
It also helps with the selection of a reference sector. We chose consumer staples based on our economic intuition of being a defensive sector.

In [ ]:
query_sector_count = sqlalchemy_text(
"""
SELECT
    s.sector,
    COUNT(a.ticker_07) AS stock_count,
    ROUND(CAST(COUNT(a.ticker_07) AS FLOAT) / (SELECT COUNT(*) FROM barra_07 WHERE nonestu_07 = 0 AND size_07 > 0) * 100, 2) AS pct_of_universe,
    ROUND(SUM(a.capitalization_07) / 1e9, 2) AS total_cap_billions,
    ROUND(AVG(a.beta_07), 3) AS avg_beta
FROM barra_07 AS a
JOIN industry_to_sector AS s ON a.indname1_07 = s.industry
WHERE a.nonestu_07 = 0 AND a.size_07 > 0
GROUP BY s.sector
ORDER BY stock_count DESC;
"""
)
with engine.connect() as connection:
    result_sector_count = pd.DataFrame(connection.execute(query_sector_count).fetchall())
print(f'{result_sector_count.shape=}')
display(result_sector_count)

In [ ]:
# sector dummies instead of industry dummies
query_sector_dummies = sqlalchemy_text(
"""
SELECT
    a.ticker_07 AS ticker,
    a.beta_07,
    a.leverage_07,
    a.voltilty_07 AS volatility_pct,
    -- Sector Dummies: omitted CONS_STAPLES as the reference
    CASE WHEN s.sector = 'FINANCIAL' THEN 1 ELSE 0 END AS d_financial,
    CASE WHEN s.sector = 'TECH_COMM' THEN 1 ELSE 0 END AS d_tech_comm,
    CASE WHEN s.sector = 'REALESTATE_UTIL' THEN 1 ELSE 0 END AS d_realestate_util,
    CASE WHEN s.sector = 'HEALTHCARE' THEN 1 ELSE 0 END AS d_healthcare,
    CASE WHEN s.sector = 'CONS_DISCR' THEN 1 ELSE 0 END AS d_cons_discr,
    CASE WHEN s.sector = 'INDUSTRIAL' THEN 1 ELSE 0 END AS d_industrial,
    CASE WHEN s.sector = 'BASIC_MAT' THEN 1 ELSE 0 END AS d_basic_mat,
    CASE WHEN s.sector = 'ENERGY' THEN 1 ELSE 0 END AS d_energy,
    CASE WHEN s.sector = 'TRANSPORT' THEN 1 ELSE 0 END AS d_transport,
    
    -- 2008 realized return
    CASE 
        WHEN b.price_08 IS NULL THEN -1.0 
        ELSE ((b.price_08 - a.price_07) / a.price_07) 
    END AS realized_return_08
FROM barra_07 AS a
JOIN industry_to_sector AS s ON a.indname1_07 = s.industry
LEFT JOIN barra_08 AS b ON a.ticker_07 = b.ticker_08
WHERE a.nonestu_07 = 0 AND a.size_07 > 0;
"""
)
with engine.connect() as connection:
    result_sector_dummies = pd.DataFrame(connection.execute(query_sector_dummies).fetchall())
print(f'{result_sector_dummies.shape=}')
display(result_sector_dummies.head(5))

### FURTHER CHANGE IN DESIGN
After checking the counts in each sector, we realized that we still have small number of stocks so we decided to combine them into:
- Financial + RealEstate_UTIL --> FIN_RE_UTIL : REASON -- these sectors are interest-rate sensitive
- INDUSTRIAL + TRANSPORT + BASIC_MAT + ENERGY --> CYCLICAL : REASON -- these sectors are macro sensitive
- CONS_STAPES + HEALTHCARE --> DEFENSIVE : REASON - a flight to safety.

In [ ]:
query_combined_sectors_dummies = sqlalchemy_text(
"""
SELECT
    a.ticker_07,
    a.beta_07,
    a.leverage_07,
    CASE WHEN s.sector IN ('FINANCIAL', 'REALESTATE_UTIL') THEN 1 ELSE 0 END AS d_fin_util,
    CASE WHEN s.sector = 'TECH_COMM' THEN 1 ELSE 0 END AS d_tech,
    CASE WHEN s.sector IN ('INDUSTRIAL', 'TRANSPORT', 'BASIC_MAT', 'ENERGY') THEN 1 ELSE 0 END AS d_cyclical,
    CASE WHEN s.sector = 'CONS_DISCR' THEN 1 ELSE 0 END AS d_discretionary,
    -- OMITTED REFERENCE: DEFENSIVE (Healthcare + Staples)
    
    CASE 
        WHEN b.price_08 IS NULL THEN -1.0 
        ELSE ((b.price_08 - a.price_07) / a.price_07) 
    END AS realized_return_08
FROM barra_07 AS a
JOIN industry_to_sector AS s ON a.indname1_07 = s.industry
LEFT JOIN barra_08 AS b ON a.ticker_07 = b.ticker_08
WHERE a.nonestu_07 = 0 AND a.size_07 > 0;
"""
)
with engine.connect() as connection:
    result_combined_sectors_dummies = pd.DataFrame(connection.execute(query_combined_sectors_dummies).fetchall())
print(f'{result_combined_sectors_dummies.shape=}')
display(result_combined_sectors_dummies.head(5))
## show combined sector dummy counts
summary = result_combined_sectors_dummies.agg({
    'd_fin_util': 'sum',
    'd_tech': 'sum',
    'd_cyclical': 'sum',
    'd_discretionary': 'sum'
})
# hard coded cons_stapes + healthcare count as defensive
defensive_count = 15 + 6
total_sum = summary.sum() + defensive_count
assert result_combined_sectors_dummies.shape[0] == total_sum, f"Expected {total_sum}, but got {result_combined_sectors_dummies.shape[0]}"

### UPDATING EXPERIMENT EXPECTATIONS
Initially we set out to validate Rosenberg and Marath's claim that our regression line would be improved from our baseline in the following ways:
- Baseline: using historical beta to predict 2008 prices.
- Approach A: adding technical factors would improve our regression line's ability to explain variance by 57%. <br>
Technical Factors are derived from price and market data
    - Volatility : price fluctuations
    - Momentum: recent price behaviors
    - Trading Activity: trading volume
- Approach B: adding only Fundamental factors would improve our regression line's ability to explain variance by 45% <br>
Fundamental Factors are derived from accounting data/statements:
    - Value: ratio of book to market - balance sheet
    - earnings variability: fluctuations in earnings and cash flows - income statement
    - leverage: debt ratios - balance sheet
- Approach C: combining both Fundamental and Technical factors would imporve our regression line's ability to explain variance by 85%.

At first we assumed they would be fairly close to these claims, but after seeing how small our universe was when we had to make a decision to pivot from industries to combined_sectors, we are anticipating that our improvements will be smaller.


In [ ]:
Image( image_folder / 'RISK_INDEX_DEF.jpg', width=700, height=400)

In [ ]:
query_model_generation_full = sqlalchemy_text(
"""
WITH Returns AS (
    SELECT
        a.ticker_07 AS ticker,
        a.barrid_07 AS barrid,
        -- Baseline Predictor
        a.hbta_07,
        -- Technical Factors (Approach A)
        a.voltilty_07,
        a.momentum_07,
        a.tradeact_07,
        -- Fundamental Factors (Approach B)
        a.value_07,
        a.earnvar_07,
        a.leverage_07,
        -- Universe & Weights
        a.capitalization_07 AS cap_07,
        s.sector AS raw_sector,
        -- 2008 Target Variable
        CASE
            WHEN b.price_08 IS NULL THEN -1.0
            ELSE ((b.price_08 - a.price_07) / a.price_07)
        END AS stock_return
    FROM barra_07 AS a
    JOIN industry_to_sector AS s ON a.indname1_07 = s.industry
    LEFT JOIN barra_08 AS b ON a.ticker_07 = b.ticker_08
    WHERE a.nonestu_07 = 0 AND a.size_07 > 0
),
Market_Calc AS (
    SELECT
        *,
        (SUM(stock_return * cap_07) OVER ()) / (SUM(cap_07) OVER ()) AS market_return
    FROM Returns
)
SELECT
    ticker,
    barrid,
    stock_return AS y_realized_return,
    hbta_07,
    -- Technicals
    voltilty_07,
    momentum_07,
    tradeact_07,
    -- Fundamentals
    value_07,
    earnvar_07,
    leverage_07,
    -- Super-Sector Dummies
    CASE WHEN raw_sector IN ('FINANCIAL', 'REALESTATE_UTIL') THEN 1 ELSE 0 END AS d_fin_util,
    CASE WHEN raw_sector = 'TECH_COMM' THEN 1 ELSE 0 END AS d_tech,
    CASE WHEN raw_sector IN ('INDUSTRIAL', 'TRANSPORT', 'BASIC_MAT', 'ENERGY') THEN 1 ELSE 0 END AS d_cyclical,
    CASE WHEN raw_sector = 'CONS_DISCR' THEN 1 ELSE 0 END AS d_discretionary
FROM Market_Calc;
"""
)
with engine.connect() as connection:
    result_model_generation_full = pd.DataFrame(connection.execute(query_model_generation_full).fetchall())

print(f'{result_model_generation_full.shape=}')
display(result_model_generation_full.head(5))

In [ ]:
SUPER_SECTORS_DUMMIES = ['d_fin_util', 'd_tech', 'd_cyclical', 'd_discretionary']
BASELINE_VARS = ['hbta_07'] + SUPER_SECTORS_DUMMIES

# APPROACH A: TECHNICAL FACTORS
TECHNICAL_VARS = ['voltilty_07', 'momentum_07', 'tradeact_07']

# APPROACH B: FUNDAMENTAL FACTORS
FUNDAMENTAL_VARS = ['value_07', 'earnvar_07', 'leverage_07']

# APPROACH C: COMBINED TECHNICAL + FUNDAMENTAL
TECHNICAL_AND_FUNDAMENTAL = TECHNICAL_VARS + FUNDAMENTAL_VARS

# Experiment
experiments = {
    "Baseline": BASELINE_VARS,
    "Approach A - Technicals": BASELINE_VARS + TECHNICAL_VARS,
    "Approach B - Fundamentals": BASELINE_VARS + FUNDAMENTAL_VARS,
    "Approach C - FUNDAMENTALS + TECHNICALS": BASELINE_VARS + TECHNICAL_AND_FUNDAMENTAL
}

In [ ]:
# forcing output to show entire OLS summary without truncation in Jupyter
display(HTML("<style>.jp-OutputArea-output {max-height: none !important;}</style>"))
audit_trail = []


for model_name, feature_list in experiments.items():
    
    # Prepare the Independent Variables (X)
    # We add a constant to represent our 'DEFENSIVE' reference sector
    X = sm.add_constant(result_model_generation_full[feature_list])
    
    # Target Variable (y)
    y = result_model_generation_full['y_realized_return']
    
    # Fit the Ordinary Least Squares (OLS) model
    model_fit = sm.OLS(y, X).fit()
    
    # Store metrics for our audit summary
    audit_trail.append({
        "Model Name": model_name,
        "R-Squared": model_fit.rsquared,
        "Adj. R-Squared": model_fit.rsquared_adj,
        "AIC": model_fit.aic,
        "Prob (F-statistic)": model_fit.f_pvalue
    })
    
    # Print the full summary for the 'Combined' model for deep-dive analysis
    if model_name == "Approach C - FUNDAMENTALS + TECHNICALS":
        print(model_fit.summary())

# 3. Create a summary DataFrame to compare performance
df_results = pd.DataFrame(audit_trail)

# Calculate % Improvement over the Baseline
baseline_r2 = df_results.iloc[0]['Adj. R-Squared']
df_results['% Imp vs Baseline'] = ((df_results['Adj. R-Squared'] - baseline_r2) / baseline_r2) * 100
df_results['Prob (F-statistic)'] = df_results['Prob (F-statistic)'].apply(lambda x: f"{x:.4e}")

display(df_results)

### 3. Transaction Management (15%)
**Scenario**: Your risk management department has a "crystal ball" predicting the 2008 crisis.

Demonstrate a transaction that:
* Identifies Bear Stearns and Lehman Brothers BARRIDs
* Temporarily adjusts their risk metrics based on "prediction"
* Analyzes impact on portfolio risk
* Uses ROLLBACK to restore original data
* Documents the what-if analysis results


In [ ]:
search_kw('bar', type_dict_07)

In [ ]:
test_query = sqlalchemy_text(
"""
SELECT
    a.ticker_07,
    a.name_07,
    a.beta_07,
    a.leverage_07,
    a.capitalization_07/1e9 AS cap_billions_07,
    a.price_07,
    a.indname1_07,
    a.nonestu_07,
    a.size_07,
    b.price_08,
    b.barrid_08,
    b.indname1_08,
    b.nonestu_08
FROM barra_07 as a
LEFT JOIN barra_08 AS b ON a.ticker_07 = b.ticker_08
WHERE a.name_07 LIKE '%BEAR%' OR a.name_07 LIKE '%LEH%';
"""
)
with engine.connect() as connection:
    test_query_result = pd.DataFrame(connection.execute(test_query).fetchall())
test_query_result

In [ ]:
query_find_bear_leh = sqlalchemy_text(
"""
SELECT
    a.ticker_07 AS ticker,
    a.price_07 AS price_07,
    a.name_07 AS name,
    a.barrid_07 AS barrid_07,
    a.indname1_07 AS industry,
    b.barrid_08 AS barrid_08,
    a.nonestu_07 AS nonestu_07
FROM barra_07 AS a
LEFT JOIN barra_08 AS b ON a.ticker_07 = b.ticker_08
JOIN industry_to_sector AS s ON a.indname1_07 = s.industry
WHERE a.nonestu_07 = 0
AND b.price_08 IS NULL
AND b.barrid_08 IS NULL
AND b.nonestu_08 IS NULL
AND s.sector = 'FINANCIAL'
ORDER BY s.industry DESC;
"""
)
with engine.connect() as connection:
    query_find_bear_leh_result = pd.DataFrame(connection.execute(query_find_bear_leh).fetchall())
query_find_bear_leh_result

In [ ]:
# creating our predictor
best_model_name = "Approach A - Technicals"
best_features = experiments[best_model_name]

# refit to get the parameters
X = sm.add_constant(result_model_generation_full[best_features])
final_model = sm.OLS(result_model_generation_full['y_realized_return'], X).fit()

# store only the weights
model_weights = final_model.params.to_dict()
model_weights

In [ ]:
# def a function that will take a stock
def predict_stock_return(stock_data_dict, model_weights):
    if model_weights is None:
        raise ValueError("Model weights are not defined.")
    prediction = model_weights['const'] # start with the constant term

    # add each weighted feature contribution
    for key, weight in model_weights.items():
        try: 
            match key:
                case 'const':
                    continue # already added
                case _:
                    descriptor_value = stock_data_dict[key]
                    prediction += (weight * descriptor_value)
        except KeyError:
            raise KeyError(f"Missing feature '{key}' in stock data for prediction.")
    return prediction

In [ ]:
# sql template for getting stock data
search_stock_barrid_query_template_approach_a = """
SELECT
    a.ticker_07, 
    a.barrid_07,
    a.hbta_07, 
    a.voltilty_07, 
    a.momentum_07, 
    a.tradeact_07,
    -- Sector Dummies for the categorical component of the model
    CASE WHEN s.sector IN ('FINANCIAL', 'REALESTATE_UTIL') THEN 1 ELSE 0 END AS d_fin_util,
    CASE WHEN s.sector = 'TECH_COMM' THEN 1 ELSE 0 END AS d_tech,
    CASE WHEN s.sector IN ('INDUSTRIAL', 'TRANSPORT', 'BASIC_MAT', 'ENERGY') THEN 1 ELSE 0 END AS d_cyclical,
    CASE WHEN s.sector = 'CONS_DISCR' THEN 1 ELSE 0 END AS d_discretionary
FROM barra_07 AS a
JOIN industry_to_sector AS s ON a.indname1_07 = s.industry
WHERE a.barrid_07 = :barrid
"""

# Convert to SQLAlchemy text object once
predict_barrid_stmt = sqlalchemy_text(search_stock_barrid_query_template_approach_a)

In [ ]:
def predict_by_barrid(target_barrid, engine, model_weights):
    with engine.connect() as connection:
        result = connection.execute(predict_barrid_stmt, {"barrid": target_barrid}).fetchone()
        if result is None:
            raise ValueError(f"No stock found with barrid {target_barrid}")
        
        stock_data = result._asdict()

    predicted_stock_return = predict_stock_return(stock_data, model_weights)

    comparison_rows = list()

    # add the intercept (const) first
    comparison_rows.append({
        'feature': 'const',
        '2007_value': 1.0, # constant term is always 1
        'weight': model_weights['const'],
        '2008_impact': model_weights['const']
    })
    # add the other descriptors
    for feature, weight in model_weights.items():
        if feature == 'const': continue
        if feature.startswith('d_'): continue # skip sector dummies for this comparison table

        value_07 = stock_data[feature]
        impact_08 = weight * value_07

        comparison_rows.append({
            'feature': feature,
            '2007_value': value_07,
            'weight': weight,
            '2008_impact': impact_08
        })
    return comparison_rows, predicted_stock_return


        
        

In [ ]:
# column mapping 2007 descriptor values to 2008 target columns
PREDICTED_COLUMN_MAPPING = {
    'price_07': 'price_08',
    'capitalization_07': 'capitalization_08',
    'voltilty_07': 'voltilty_08',
    'momentum_07': 'momentum_08',
    'tradeact_07': 'tradeact_08',
}

In [ ]:
# build payload 
def build_update_payload(barrid, features, total_return, engine, old_to_new_col_mapping):
    payload = {'barrid': barrid}

    with engine.connect() as connection:
        row_07 = connection.execute(
            sqlalchemy_text("""
                            SELECT
                                price_07,
                                capitalization_07
                             FROM barra_07 WHERE barrid_07 = :barrid
                            """),
                            {"barrid": barrid}
        ).fetchone()
    
    if row_07:
        price_07 = row_07.price_07
        cap_07 = row_07.capitalization_07

    # calculate price
    payload['price_08'] = price_07 * (1 + total_return)

    # calculate cap
    payload['capitalization_08'] = cap_07 * (1 + total_return)

    # extract metrics
    for feature in features:
        feature_name = feature['feature']
        if feature_name == 'hbta_07':
            continue # skip hbta as it is a predictor, not a target
        if feature_name in old_to_new_col_mapping:
            new_col = old_to_new_col_mapping[feature_name]
            payload[new_col] = feature['2008_impact']
    return payload

In [ ]:
# get data and build payload for bear
# bear : barrid USABM91


bear_metrics, bear_prediction = predict_by_barrid('USABM91', engine, model_weights)

bear_payload = build_update_payload(
    barrid='USABM91',
    features=bear_metrics,
    total_return=bear_prediction,
    engine=engine,
    old_to_new_col_mapping=PREDICTED_COLUMN_MAPPING
)
bear_payload

In [ ]:
# get data and build payload for lehman
# lehman : barrid USARHS2

lehman_metrics, lehman_prediction = predict_by_barrid('USARHS2', engine, model_weights)

lehman_payload = build_update_payload(
    barrid='USARHS2',
    features=lehman_metrics,
    total_return=lehman_prediction,
    engine=engine,
    old_to_new_col_mapping=PREDICTED_COLUMN_MAPPING
)
lehman_payload

#### Analyze impact on portfolio risk
For our portfolio we are going to use the stocks that belong to the barra estimation universe (nonestu = 0) in 2007. <br>
If we had chosen size > 0 as an additional filter, bear stearns would be excluded.

In [ ]:
# create a view of our portfolio
query_create_nonestu_portfolio_view_2007 = sqlalchemy_text(
"""
CREATE VIEW IF NOT EXISTS v_nonestu_portfolio_2007 AS
SELECT
    a.ticker_07,
    a.barrid_07,
    a.price_07,
    a.voltilty_07,
    a.tradeact_07,
    a.momentum_07,
    a.srisk_pct_07,
    a.trisk_pct_07,
    a.capitalization_07,
    a.indname1_07,
    a.beta_07
FROM barra_07 AS a
JOIN industry_to_sector AS s ON a.indname1_07 = s.industry
WHERE a.nonestu_07 = 0;
""")

check_if_nonestu_portfolio_view_exists = sqlalchemy_text("""
SELECT name FROM sqlite_master WHERE type='view' AND name='v_nonestu_portfolio_2007';
""")

with engine.connect() as connection:
    check_if_nonestu_portfolio_view_exists = connection.execute(check_if_nonestu_portfolio_view_exists).fetchone() is not None
    connection.execute( query_create_nonestu_portfolio_view_2007)
    
    if check_if_nonestu_portfolio_view_exists:
        logger.info('view already exists. sql query executed without error.')
    else:
        logger.info('view created successfully.')

### Compute Portfolio Risk
- Portfolio weights - from our 2007 view, nonestu = 0
- Factor exposures - volatility, tradeactivity, momentum
- factor returns - coefficients from our regression using approach A
- portfolio factor exposure - weighted average of factor exposures
- Factor covariance matrix - derived from our three factors (vol, tradeact, mom)
- Residual returns - 


In [ ]:
query_portfolio_weights_and_risk_exposure_07 = sqlalchemy_text(
"""
--- Portfolio Weights and Residual Risk Calculation
WITH Portfolio AS (
SELECT
    *, -- because we created a view with only the relevant universe, we can safely select * here for ease of use
    -- Calculate Total cap for weights
    SUM(capitalization_07) OVER () AS total_portfolio_cap,
    -- Calculate Individual Residual Risk
    SQRT(POWER(trisk_pct_07, 2) - POWER(srisk_pct_07, 2)) AS residual_risk_pct_07
FROM v_nonestu_portfolio_2007
),
Weights AS (
SELECT
    *,
    (capitalization_07 / total_portfolio_cap) AS portfolio_weight
FROM Portfolio
), --  Portfolio Factor Exposures
PorfolioExposures AS (
SELECT
    SUM(portfolio_weight * voltilty_07) AS weighted_volatility,
    SUM(portfolio_weight * tradeact_07) AS weighted_tradeact,
    SUM(portfolio_weight * momentum_07) AS weighted_momentum,
    --- portfolio residual variance
    SUM(POWER(portfolio_weight,2) * POWER(residual_risk_pct_07,2)) AS portfolio_residual_variance
FROM Weights
), --- Factor Covariance Matrix (F)
FactorCovariance AS (
SELECT
    -- Variance/Covariance for volatility, trade activity, and momentum
    COVAR_SAMP(voltilty_07, voltilty_07) AS var_volatility,
    COVAR_SAMP(tradeact_07, tradeact_07) AS var_tradeact,
    COVAR_SAMP(momentum_07, momentum_07) AS var_momentum,
    COVAR_SAMP(voltilty_07, tradeact_07) AS cov_vol_trade,
    COVAR_SAMP(voltilty_07, momentum_07) AS cov_vol_mom,
    COVAR_SAMP(tradeact_07, momentum_07) AS cov_trade_mom
FROM Weights
)
--- Systematic and Total Risk
SELECT
(
    (POWER(weighted_volatility, 2) * var_volatility) +
    (POWER(weighted_tradeact, 2) * var_tradeact) +
    (POWER(weighted_momentum, 2) * var_momentum) +
    (2 * weighted_volatility * weighted_tradeact * cov_vol_trade) +
    (2 * weighted_volatility * weighted_momentum * cov_vol_mom) +
    (2 * weighted_tradeact * weighted_momentum * cov_trade_mom)
) * 10000 AS scaled_by_10e4_portfolio_systematic_variance,
portfolio_residual_variance,
-- Total Risk
SQRT(
    (POWER(weighted_volatility, 2) * var_volatility) + (POWER(weighted_tradeact, 2) * var_tradeact) + (POWER(weighted_momentum, 2) * var_momentum) +
    (2 * weighted_volatility * weighted_tradeact * cov_vol_trade) + (2 * weighted_volatility * weighted_momentum * cov_vol_mom) +
    (2 * weighted_tradeact * weighted_momentum * cov_trade_mom) +
    portfolio_residual_variance
) AS portfolio_total_risk
FROM PorfolioExposures, FactorCovariance;
"""
)
with engine.connect() as connection:
    result_portfolio_weights_and_risk_exposure = pd.DataFrame(connection.execute(query_portfolio_weights_and_risk_exposure_07))
print(f'{result_portfolio_weights_and_risk_exposure.shape=}')
result_portfolio_weights_and_risk_exposure.head(5)

In [ ]:
# debugging systematic variance 0
query_debug_systematic_variance_07 = sqlalchemy_text(
"""
WITH Portfolio AS (
    SELECT *, SUM(capitalization_07) OVER () AS total_portfolio_cap
    FROM v_nonestu_portfolio_2007
),
Weights AS (
    SELECT *, (capitalization_07 / total_portfolio_cap) AS portfolio_weight
    FROM Portfolio
)
SELECT 
    -- Are the tilts 0?
    SUM(portfolio_weight * voltilty_07) AS check_tilt,
    -- Is the variance 0?
    COVAR_SAMP(voltilty_07, voltilty_07) AS check_var,
    -- How many rows are we actually seeing?
    COUNT(*) AS row_count
FROM Weights;
""")
with engine.connect() as connection:
    result_debug_systematic_variance_07 = pd.DataFrame(connection.execute(query_debug_systematic_variance_07))
result_debug_systematic_variance_07

In [ ]:
query_create_nonestu_portfolio_view_2008 = sqlalchemy_text(
"""
CREATE VIEW IF NOT EXISTS v_nonestu_portfolio_2008 AS
SELECT
    a.ticker_08,
    a.barrid_08,
    a.price_08,
    a.voltilty_08,
    a.tradeact_08,
    a.momentum_08,
    a.srisk_pct_08,
    a.trisk_pct_08,
    a.capitalization_08,
    a.indname1_08,
    a.beta_08
FROM barra_08 AS a
JOIN industry_to_sector AS s ON a.indname1_08 = s.industry
WHERE a.nonestu_08 = 0;
"""
)
check_if_nonestu_portfolio_view_08_exists = sqlalchemy_text("""
SELECT name FROM sqlite_master WHERE type='view' AND name='v_nonestu_portfolio_2008';
""")

with engine.connect() as connection:
    check_if_nonestu_portfolio_view_08_exists = connection.execute(check_if_nonestu_portfolio_view_08_exists).fetchone() is not None
    connection.execute( query_create_nonestu_portfolio_view_2008)
    
    if check_if_nonestu_portfolio_view_08_exists:
        logger.info('view already exists. sql query executed without error.')
    else:
        logger.info('view created successfully.')



In [ ]:
query_portfolio_weights_and_risk_exposure_08 = sqlalchemy_text(
"""
--- Portfolio Weights and Residual Risk Calculation
WITH Portfolio AS (
SELECT
    *, -- because we created a view with only the relevant universe, we can safely select * here for ease of use
    -- Calculate Total cap for weights
    SUM(capitalization_08) OVER () AS total_portfolio_cap,
    -- Calculate Individual Residual Risk
    SQRT(POWER(trisk_pct_08, 2) - POWER(srisk_pct_08, 2)) AS residual_risk_pct_08
FROM v_nonestu_portfolio_2008
),
Weights AS (
SELECT
    *,
    (capitalization_08 / total_portfolio_cap) AS portfolio_weight
FROM Portfolio
), --  Portfolio Factor Exposures
PorfolioExposures AS (
SELECT
    SUM(portfolio_weight * voltilty_08) AS weighted_volatility,
    SUM(portfolio_weight * tradeact_08) AS weighted_tradeact,
    SUM(portfolio_weight * momentum_08) AS weighted_momentum,
    --- portfolio residual variance
    SUM(POWER(portfolio_weight,2) * POWER(residual_risk_pct_08,2)) AS portfolio_residual_variance
FROM Weights
), --- Factor Covariance Matrix (F)
FactorCovariance AS (
SELECT
    -- Variance/Covariance for volatility, trade activity, and momentum
    COVAR_SAMP(voltilty_08, voltilty_08) AS var_volatility,
    COVAR_SAMP(tradeact_08, tradeact_08) AS var_tradeact,
    COVAR_SAMP(momentum_08, momentum_08) AS var_momentum,
    COVAR_SAMP(voltilty_08, tradeact_08) AS cov_vol_trade,
    COVAR_SAMP(voltilty_08, momentum_08) AS cov_vol_mom,
    COVAR_SAMP(tradeact_08, momentum_08) AS cov_trade_mom
FROM Weights
)
--- Systematic and Total Risk
SELECT
(
    (POWER(weighted_volatility, 2) * var_volatility) +
    (POWER(weighted_tradeact, 2) * var_tradeact) +
    (POWER(weighted_momentum, 2) * var_momentum) +
    (2 * weighted_volatility * weighted_tradeact * cov_vol_trade) +
    (2 * weighted_volatility * weighted_momentum * cov_vol_mom) +
    (2 * weighted_tradeact * weighted_momentum * cov_trade_mom)
) * 10000 AS scaled_by_10e4_portfolio_systematic_variance,
portfolio_residual_variance,
-- Total Risk
SQRT(
    (POWER(weighted_volatility, 2) * var_volatility) + (POWER(weighted_tradeact, 2) * var_tradeact) + (POWER(weighted_momentum, 2) * var_momentum) +
    (2 * weighted_volatility * weighted_tradeact * cov_vol_trade) + (2 * weighted_volatility * weighted_momentum * cov_vol_mom) +
    (2 * weighted_tradeact * weighted_momentum * cov_trade_mom) +
    portfolio_residual_variance
) AS portfolio_total_risk
FROM PorfolioExposures, FactorCovariance;
"""
)
with engine.connect() as connection:
    result_portfolio_weights_and_risk_exposure_08 = pd.DataFrame(connection.execute(query_portfolio_weights_and_risk_exposure_08))
print(f'{result_portfolio_weights_and_risk_exposure_08.shape=}')
result_portfolio_weights_and_risk_exposure_08.head(5)

In [ ]:
# debugging systematic variance 0
query_debug_systematic_variance_08 = sqlalchemy_text(
"""
WITH Portfolio AS (
    SELECT *, SUM(capitalization_08) OVER () AS total_portfolio_cap
    FROM v_nonestu_portfolio_2008
),
Weights AS (
    SELECT *, (capitalization_08 / total_portfolio_cap) AS portfolio_weight
    FROM Portfolio
)
SELECT 
    -- Are the tilts 0?
    SUM(portfolio_weight * voltilty_08) AS check_tilt,
    -- Is the variance 0?
    COVAR_SAMP(voltilty_08, voltilty_08) AS check_var,
    -- How many rows are we actually seeing?
    COUNT(*) AS row_count
FROM Weights;
""")
with engine.connect() as connection:
    result_debug_systematic_variance_08 = pd.DataFrame(connection.execute(query_debug_systematic_variance_08))
result_debug_systematic_variance_08

### To calculate portfolio risk for 2008
we need trisk and srisk.
We will handle in two ways:
1. base case: using 2007 estimates
2. stressed case: derive Financial Sector multiplier and apply

**CONSIDERATIONS**<br>
we will use median so that can find out what a typical surivor in 2008 looked like for the financial sector. if we use average, it might be understated due to the only healthy firms having survived. We won't apply a distressed premium for lehman and bear to simplify things.


In [ ]:
# sector stress multiplier
query_stress_multiplier_financial_sector = sqlalchemy_text(
"""
SELECT
    MEDIAN(b.trisk_pct_08 / a.trisk_pct_07) AS financial_sector_stress_multiplier
    FROM barra_08 AS b
    JOIN barra_07 AS a ON b.ticker_08 = a.ticker_07
    JOIN industry_to_sector AS s ON a.indname1_07 = s.industry
    WHERE s.sector = 'FINANCIAL'
        AND a.nonestu_07 = 0
        AND b.nonestu_08 = 0
        AND a.trisk_pct_07 > 0
        AND b.trisk_pct_08 IS NOT NULL
        AND financial_sector_stress_multiplier BETWEEN 0.5 AND 5.0; -- filter out extreme outliers that may skew the median;
""")
